https://www.kaggle.com/theoviel/deep-learning-starter-simple-lstm/notebook

# Load libraries

In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "wandb"
secret_value = UserSecretsClient().get_secret(secret_label)
!wandb login $secret_value

In [ ]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# from tqdm.notebook import tqdm
from tqdm import tqdm
from collections import Counter

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import RobustScaler

import torch

warnings.filterwarnings("ignore")
NUM_WORKERS = os.cpu_count()

# Configs

In [ ]:
class Config:
    """
    Parameters used for training
    """
    # wandb params
    EXP_NAME = "gbvpp_starter"
    OUTPUT = "/kaggle/working"
    
    # General
    seed = 42
    verbose = 1
    device = "cuda" if torch.cuda.is_available() else "cpu"
    save_weights = True

    # k-fold
    k = 5
    selected_folds = [0, 1, 2, 3, 4]
    
    # Model
    selected_model = 'rnn'
    input_dim = 38

    dense_dim = 512
    lstm_dim1 = 1024
    lstm_dim2 = 512
    lstm_dim3 = 256
    logit_dim = 512
    num_classes = 1

    # Training
    loss = "L1Loss"  # not used
    optimizer = "Adam"
    batch_size = 256
    epochs = 200

    lr = 1e-3
    warmup_prop = 0

    val_bs = 256
    first_epoch_eval = 0

# Explore data

In [ ]:
# DEBUG = True

DATA_PATH = "/kaggle/input/ventilator-pressure-prediction/"

sub = pd.read_csv(DATA_PATH + 'sample_submission.csv')
df_train = pd.read_csv(DATA_PATH + 'train.csv')
df_test = pd.read_csv(DATA_PATH + 'test.csv')

# if DEBUG:
df = df_train[df_train['breath_id'] < 5].reset_index(drop=True)

In [ ]:
print(f'Length of TRAIN dataset: {len(df_train)}')
print(f'Length of TEST dataset: {len(df_test)}')
print('')
print(f'Number of breaths in train dataset: {df_train["breath_id"].nunique()}')
print(f'Number of breaths in test dataset: {df_test["breath_id"].nunique()}')
print(f'The number of observations for each breath: {df_train["breath_id"].value_counts().reset_index()["breath_id"].unique()[0]}')

# Feature Engineering

In [ ]:
def add_features(df):
    df['area'] = df['time_step'] * df['u_in']
    df['area'] = df.groupby('breath_id')['area'].cumsum()
    
    df["u_in_sum"] = df.groupby("breath_id")["u_in"].transform("sum")
    df["u_out_sum"] = df.groupby("breath_id")["u_out"].transform("sum")
    
    df['u_in_cumsum'] = (df['u_in']).groupby(df['breath_id']).cumsum()
    
    df['u_in_lag1'] = df.groupby('breath_id')['u_in'].shift(1)
#     df['u_out_lag1'] = df.groupby('breath_id')['u_out'].shift(1)
#     df['u_in_lag_back1'] = df.groupby('breath_id')['u_in'].shift(-1)
#     df['u_out_lag_back1'] = df.groupby('breath_id')['u_out'].shift(-1)
    
    df['u_in_lag2'] = df.groupby('breath_id')['u_in'].shift(2)
#     df['u_out_lag2'] = df.groupby('breath_id')['u_out'].shift(2)
#     df['u_in_lag_back2'] = df.groupby('breath_id')['u_in'].shift(-2)
#     df['u_out_lag_back2'] = df.groupby('breath_id')['u_out'].shift(-2)
    
    df['u_in_lag3'] = df.groupby('breath_id')['u_in'].shift(3)
#     df['u_out_lag3'] = df.groupby('breath_id')['u_out'].shift(3)
#     df['u_in_lag_back3'] = df.groupby('breath_id')['u_in'].shift(-3)
#     df['u_out_lag_back3'] = df.groupby('breath_id')['u_out'].shift(-3)
    
    df['u_in_lag4'] = df.groupby('breath_id')['u_in'].shift(4)
#     df['u_out_lag4'] = df.groupby('breath_id')['u_out'].shift(4)
#     df['u_in_lag_back4'] = df.groupby('breath_id')['u_in'].shift(-4)
#     df['u_out_lag_back4'] = df.groupby('breath_id')['u_out'].shift(-4)
    df = df.fillna(0)
    
    df['breath_id__u_in__max'] = df.groupby(['breath_id'])['u_in'].transform('max')
#     df['breath_id__u_out__max'] = df.groupby(['breath_id'])['u_out'].transform('max')
    
    df['u_in_diff1'] = df['u_in'] - df['u_in_lag1']
#     df['u_out_diff1'] = df['u_out'] - df['u_out_lag1']
    df['u_in_diff2'] = df['u_in'] - df['u_in_lag2']
#     df['u_out_diff2'] = df['u_out'] - df['u_out_lag2']
    
    df['breath_id__u_in__diffmax'] = df.groupby(['breath_id'])['u_in'].transform('max') - df['u_in']
    df['breath_id__u_in__diffmean'] = df.groupby(['breath_id'])['u_in'].transform('mean') - df['u_in']
    
    df['breath_id__u_in__diffmax'] = df.groupby(['breath_id'])['u_in'].transform('max') - df['u_in']
    df['breath_id__u_in__diffmean'] = df.groupby(['breath_id'])['u_in'].transform('mean') - df['u_in']
    
    df['u_in_diff3'] = df['u_in'] - df['u_in_lag3']
#     df['u_out_diff3'] = df['u_out'] - df['u_out_lag3']
    df['u_in_diff4'] = df['u_in'] - df['u_in_lag4']
#     df['u_out_diff4'] = df['u_out'] - df['u_out_lag4']
    df['cross']= df['u_in']*df['u_out']
#     df['cross2']= df['time_step']*df['u_out']
    
    df["u_in_rolling_mean10"] = df[["breath_id", "u_in"]].groupby("breath_id").rolling(10).mean()["u_in"].reset_index(drop=True)    
    df["u_in_rolling_max10"] = df[["breath_id", "u_in"]].groupby("breath_id").rolling(10).max()["u_in"].reset_index(drop=True)
    df["u_in_rolling_min10"] = df[["breath_id", "u_in"]].groupby("breath_id").rolling(10).min()["u_in"].reset_index(drop=True)
    df["u_in_rolling_std10"] = df[["breath_id", "u_in"]].groupby("breath_id").rolling(10).std()["u_in"].reset_index(drop=True)

    df["time_passed"] = df.groupby("breath_id")["time_step"].diff()
    
    df = df.fillna(0)
    
    df['R'] = df['R'].astype(str)
    df['C'] = df['C'].astype(str)
    df['R__C'] = df["R"].astype(str) + '__' + df["C"].astype(str)
    df = pd.get_dummies(df)
    return df

In [ ]:
print(df_train.shape)
print(df_test.shape)

df_train = add_features(df_train)
df_test = add_features(df_test)

dropCols = ['time_step']

df_train.drop(columns=dropCols, inplace=True)
df_test.drop(columns=dropCols, inplace=True)

print(df_train.shape)
print(df_test.shape)

In [ ]:
df_train.columns

- https://www.kaggle.com/c/ventilator-pressure-prediction/discussion/274196
- https://www.kaggle.com/c/ventilator-pressure-prediction/discussion/273974
- https://www.kaggle.com/patrick0302/add-last-u-in-as-new-feat
- https://www.kaggle.com/c/ventilator-pressure-prediction/discussion/276409

# Normalization

In [ ]:
RS = RobustScaler()

trainCols = df_train.drop(columns=['id','breath_id','pressure','u_out']).columns
testCols = df_test.drop(columns=['id','breath_id','u_out']).columns

tgt_train = df_train[['u_out','id','breath_id','pressure']].copy()
tgt_test = df_test[['u_out','id','breath_id']].copy()

df_train = RS.fit_transform(df_train.drop(columns=['u_out','id','breath_id','pressure']))
df_test = RS.transform(df_test.drop(columns=['u_out','id','breath_id']))

df_train = pd.DataFrame(df_train, columns=trainCols)
df_test = pd.DataFrame(df_test, columns=testCols)

df_train = pd.concat([tgt_train, df_train], axis=1)
df_test = pd.concat([tgt_test, df_test], axis=1)

print(df_train.shape)
print(df_test.shape)

In [ ]:
# df_train.to_feather("train.feather")
# df_test.to_feather("test.feather")

# Read in clean data

In [ ]:
# df_train = pd.read_feather("../input/gbvpptraindata/train.feather")
# df_test = pd.read_feather("../input/gbvpptraindata/test.feather")
# df = df_train[df_train['breath_id'] < 5].reset_index(drop=True)

# print(df_train.shape)
# print(df_test.shape)
# print(df.shape)

# Dataset

In [ ]:
import torch
from torch.utils.data import Dataset

class VentilatorDataset(Dataset):
    def __init__(self, df):
        if "pressure" not in df.columns:
            df['pressure'] = 0

        self.df = df.groupby('breath_id').agg(list).reset_index()
        
        self.prepare_data()
                
    def __len__(self):
        return self.df.shape[0]
    
    def prepare_data(self):
        self.pressures = np.array(self.df['pressure'].values.tolist())
        
        rs = np.array(self.df['R'].values.tolist())
        cs = np.array(self.df['C'].values.tolist())
        u_ins = np.array(self.df['u_in'].values.tolist())
        
        self.u_outs = np.array(self.df['u_out'].values.tolist())
        
        self.inputs = np.concatenate([
            rs[:, None], 
            cs[:, None], 
            u_ins[:, None], 
            np.cumsum(u_ins, 1)[:, None],
            self.u_outs[:, None]
        ], 1).transpose(0, 2, 1)

    def __getitem__(self, idx):
        data = {
            "input": torch.tensor(self.inputs[idx], dtype=torch.float),
            "u_out": torch.tensor(self.u_outs[idx], dtype=torch.float),
            "p": torch.tensor(self.pressures[idx], dtype=torch.float),
        }
        
        return data

# Model architecture

In [ ]:
import torch
import torch.nn as nn


class BiLSTMModel(nn.Module):
    def __init__(
        self,
        input_dim=4,
        lstm_dim1=256,
        lstm_dim2=256,
        lstm_dim3=256,
        dense_dim=256,
        logit_dim=256,
        num_classes=1,
    ):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, dense_dim),
            nn.ReLU(),
#             nn.Linear(dense_dim // 2, dense_dim),
#             nn.ReLU(),
        )

        self.lstm1 = nn.LSTM(dense_dim, lstm_dim1, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(lstm_dim1 * 2, lstm_dim2, batch_first=True, bidirectional=True)
        self.lstm3 = nn.LSTM(lstm_dim2 * 2, lstm_dim3, batch_first=True, bidirectional=True)

        self.logits = nn.Sequential(
            nn.Linear(lstm_dim3 * 2, logit_dim),
            nn.ReLU(),
            nn.Linear(logit_dim, num_classes),
        )

    def forward(self, x):
        features = self.mlp(x)
        features, _ = self.lstm1(features)
        features, _ = self.lstm2(features)
        features, _ = self.lstm3(features)
        pred = self.logits(features)
        return pred

# Utils

In [ ]:
import os
import torch
import random
import numpy as np


def seed_everything(seed):
    """
    Seeds basic parameters for reproductibility of results.

    Args:
        seed (int): Number of the seed.
    """
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    
def count_parameters(model, all=False):
    """
    Counts the parameters of a model.

    Args:
        model (torch model): Model to count the parameters of.
        all (bool, optional):  Whether to count not trainable parameters. Defaults to False.

    Returns:
        int: Number of parameters.
    """
    if all:
        return sum(p.numel() for p in model.parameters())
    else:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    
def worker_init_fn(worker_id):
    """
    Handles PyTorch x Numpy seeding issues.

    Args:
        worker_id (int): Id of the worker.
    """
    np.random.seed(np.random.get_state()[1][0] + worker_id)
    

def save_model_weights(model, filename, verbose=1, cp_folder=""):
    """
    Saves the weights of a PyTorch model.

    Args:
        model (torch model): Model to save the weights of.
        filename (str): Name of the checkpoint.
        verbose (int, optional): Whether to display infos. Defaults to 1.
        cp_folder (str, optional): Folder to save to. Defaults to "".
    """
    if verbose:
        print(f"\n -> Saving weights to {os.path.join(cp_folder, filename)}\n")
    torch.save(model.state_dict(), os.path.join(cp_folder, filename))

# Competition metric

In [ ]:
def compute_metric(df, preds):
    """
    Metric for the problem, as I understood it.
    """
    df = df.groupby('breath_id').agg(list).reset_index()

    y = np.array(df['pressure'].values.tolist())
    w = 1 - np.array(df['u_out'].values.tolist())

    assert y.shape == preds.shape and w.shape == y.shape, (y.shape, preds.shape, w.shape)

    mae = w * np.abs(y - preds)
    mae = mae.sum() / w.sum()

    return mae


class VentilatorLoss(nn.Module):
    """
    Directly optimizes the competition metric
    """
    def __call__(self, preds, y, u_out):
        w = 1 - u_out
        mae = w * (y - preds).abs()
        mae = mae.sum(-1) / w.sum(-1)

        return mae

# Model fitting

In [ ]:
import gc
import time
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup


def fit(
    model,
    fold,
    df_val,
    train_dataset,
    val_dataset,
    loss_name="L1Loss",
    optimizer="Adam",
    epochs=50,
    batch_size=32,
    val_bs=32,
    warmup_prop=0.1,
    lr=1e-3,
    num_classes=1,
    verbose=1,
    first_epoch_eval=0,
    device="cuda"
):
    best_val_loss = 10.
    latest_model = ""
    avg_val_loss = 0.

    # Optimizer
    optimizer = getattr(torch.optim, optimizer)(model.parameters(), lr=lr)

    # Data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        worker_init_fn=worker_init_fn
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=val_bs,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    # Loss
#     loss_fct = getattr(torch.nn, loss_name)(reduction="none")
    loss_fct = VentilatorLoss()

    # Scheduler
    num_warmup_steps = int(warmup_prop * epochs * len(train_loader))
    num_training_steps = int(epochs * len(train_loader))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps, num_training_steps
    )

    for epoch in range(epochs):
        model.train()
        model.zero_grad()
        start_time = time.time()

        avg_loss = 0
        for data in train_loader:
            pred = model(data[0].to(device)).squeeze(-1)
            

            loss = loss_fct(
                    pred.to("cuda"),
                    data[1].squeeze(-1).to("cuda"),
                    data[0][:,:,0].reshape(-1,80,1).squeeze(-1).to("cuda"),
                ).mean()
            loss.backward()
            avg_loss += loss.item() / len(train_loader)

            optimizer.step()
            scheduler.step()

            for param in model.parameters():
                param.grad = None

        model.eval()
        mae, avg_val_loss = 0, 0
        preds = []

        with torch.no_grad():
            for data in val_loader:
                pred = model(data[0].to(device)).squeeze(-1)

                loss = loss_fct(
                    pred.to("cuda"),
                    data[1].squeeze(-1).to("cuda"),
                    data[0][:,:,0].reshape(-1,80,1).squeeze(-1).to("cuda"),
                ).mean()
                avg_val_loss += loss.item() / len(val_loader)

                preds.append(pred.detach().cpu().numpy())
        
        preds = np.concatenate(preds, 0)
        mae = compute_metric(df_val, preds)

        elapsed_time = time.time() - start_time
        if (epoch + 1) % verbose == 0:
            elapsed_time = elapsed_time * verbose
            lr = scheduler.get_last_lr()[0]
            print(
                f"Epoch {epoch + 1:02d}/{epochs:02d} \t lr={lr:.1e}\t t={elapsed_time:.0f}s \t"
                f"loss={avg_loss:.3f}",
                end="\t",
            )

            if (epoch + 1 >= first_epoch_eval) or (epoch + 1 == epochs):
                print(f"val_loss={avg_val_loss:.3f}\tmae={mae:.3f}")
                
                if avg_val_loss < best_val_loss:
                    
                    print("\n Performance has improved; Saving model \n")
                    
                    if latest_model != "":
                        !rm *{fold}*.pt
                
                    save_model_weights(
                                model,
                                f"biLSTM_{fold}_mae{int(mae*1000)}.pt",
                                cp_folder="",
                            )
                    latest_model = f"biLSTM_{fold}_mae{int(mae*1000)}.pt"
                    best_val_loss = avg_val_loss 
            else:
                print("")

    del (val_loader, train_loader, loss, data, pred)
    gc.collect()
    torch.cuda.empty_cache()
    
    # Use best model
    model.load_state_dict(torch.load(latest_model))
    
    model.eval()
    
    preds = []

    with torch.no_grad():
        for data in val_loader:
            pred = model(data[0].to(device)).squeeze(-1)

            loss = loss_fct(
                    pred.to("cuda"),
                    data[1].squeeze(-1).to("cuda"),
                    data[0][:,:,0].reshape(-1,80,1).squeeze(-1).to("cuda"),
                ).mean()
            avg_val_loss += loss.item() / len(val_loader)

            preds.append(pred.detach().cpu().numpy())

    preds = np.concatenate(preds, 0)
    mae = compute_metric(val_dataset.df, preds)
    
    print(f"FOLD {fold} MAE={mae:.3f}")

    return preds, latest_model

# Understand the fit function

In [ ]:
# trainCols

In [ ]:
# train_dataset = torch.utils.data.TensorDataset(
#         torch.from_numpy(np.float32(df_train[["u_out"]+trainCols.tolist()]).reshape(-1, 80, len(trainCols)+1)),
#         torch.from_numpy(np.float32(df_train['pressure']).reshape(-1, 80, 1))
#     )

In [ ]:
# train_loader = DataLoader(
#         train_dataset,
#         batch_size=512,
#         shuffle=True,
#         drop_last=False,
#         num_workers=4,
#         pin_memory=True,
#         worker_init_fn=worker_init_fn
#     )

In [ ]:
# seed_everything(Config.seed)

# model = RNNModel(
#     input_dim=16,
#     lstm_dim=Config.lstm_dim,
#     dense_dim=Config.dense_dim,
#     logit_dim=Config.logit_dim,
#     num_classes=Config.num_classes,
# ).to(Config.device)
# model.zero_grad()

In [ ]:
# model.train()
# model.zero_grad()
# avg_loss = 0

# loss_fct = VentilatorLoss()

# avg_val_loss=0

# model(data[0].to("cuda")).squeeze(-1)

In [ ]:
# model.train()
# model.zero_grad()
# avg_loss = 0

# loss_fct = VentilatorLoss()

# avg_val_loss=0

# preds =[]

# for data in tqdm(train_loader):
#     pred = model(data[0].to("cuda")).squeeze(-1)

#     loss = loss_fct(
#                     pred.to("cuda"),
#                     data[1].squeeze(-1).to("cuda"),
#                     data[0][:,:,0].reshape(-1,80,1).squeeze(-1).to("cuda"),
#                 ).mean()
#     avg_val_loss += loss.item() / len(train_loader)
    
#     preds.append(pred.detach().cpu().numpy())


# preds = np.concatenate(preds, 0)
# mae = compute_metric(df_train, preds)
    

In [ ]:
# print(avg_val_loss)
# print(mae)

In [ ]:
# u_out = data[0][:,:,0].reshape(-1,80,1).squeeze(-1).to("cuda")
# y = data[1].squeeze(-1).to("cuda")
# preds_tst = pred.reshape(-1,80,1).squeeze(-1).to("cuda")

# print(f"u_out shape: {u_out.shape}")
# print(f"y shape: {y.shape}")
# print(f"preds_tst shape: {preds_tst.shape}")


# w = 1 - u_out
# mae = w * (y - preds_tst).abs()
# mae = mae.sum(-1) / w.sum(-1)

# print(w.shape)
# print(mae.shape)

# Model training

In [ ]:
def train(config, df_train, df_val, df_test, fold):
    """
    Trains and validate a model.

    Args:
        config (Config): Parameters.
        df_train (pandas dataframe): Training metadata.
        df_val (pandas dataframe): Validation metadata.
        df_test (pandas dataframe): Test metadata.
        fold (int): Selected fold.

    Returns:
        np array: Study validation predictions.
    """

    seed_everything(config.seed)

    model = BiLSTMModel(
        input_dim=config.input_dim,
        lstm_dim1=config.lstm_dim1,
        lstm_dim2=config.lstm_dim2,
        lstm_dim3=config.lstm_dim3,
        dense_dim=config.dense_dim,
        logit_dim=config.logit_dim,
        num_classes=config.num_classes,
    ).to(config.device)
    model.zero_grad()

#     train_dataset = VentilatorDataset(df_train)
#     val_dataset = VentilatorDataset(df_val)
#     test_dataset = VentilatorDataset(df_test)

    trainCols = ["u_out"] + df_train.drop(columns=['id','breath_id','pressure','u_out']).columns.tolist()
    testCols = ["u_out"] + df_test.drop(columns=['id','breath_id','u_out']).columns.tolist()
    
    train_dataset = torch.utils.data.TensorDataset(
        torch.from_numpy(np.float32(df_train[trainCols]).reshape(-1, 80, len(trainCols))),
        torch.from_numpy(np.float32(df_train['pressure']).reshape(-1, 80, 1))
    )
    val_dataset = torch.utils.data.TensorDataset(
        torch.from_numpy(np.float32(df_val[trainCols]).reshape(-1, 80, len(trainCols))),
        torch.from_numpy(np.float32(df_val['pressure']).reshape(-1, 80, 1))
    )
    test_dataset = torch.from_numpy(np.float32(df_test[testCols]).reshape(-1, 80, len(testCols)))

    n_parameters = count_parameters(model)

    print(f"    -> {len(train_dataset)} training breathes")
    print(f"    -> {len(val_dataset)} validation breathes")
    print(f"    -> {n_parameters} trainable parameters\n")

    pred_val, fold_model = fit(
        model,
        fold,
        df_val,
        train_dataset,
        val_dataset,
        loss_name=config.loss,
        optimizer=config.optimizer,
        epochs=config.epochs,
        batch_size=config.batch_size,
        val_bs=config.val_bs,
        lr=config.lr,
        warmup_prop=config.warmup_prop,
        verbose=config.verbose,
        first_epoch_eval=config.first_epoch_eval,
        device=config.device,
    )
    
    pred_test = predict(
        fold_model, 
        test_dataset, 
        batch_size=config.val_bs, 
        device=config.device
    )

#     if config.save_weights:
#         save_model_weights(
#             model,
#             f"{config.selected_model}_{fold}.pt",
#             cp_folder="",
#         )

    del (model, train_dataset, val_dataset, test_dataset)
    gc.collect()
    torch.cuda.empty_cache()

    return pred_val, pred_test

# Model predict

In [ ]:
def predict(
    latest_model,
    dataset,
    batch_size=64,
    device="cuda"
):
    """
    Usual torch predict function. Supports sigmoid and softmax activations.
    Args:
        model (torch model): Model to predict with.
        dataset (PathologyDataset): Dataset to predict on.
        batch_size (int, optional): Batch size. Defaults to 64.
        device (str, optional): Device for torch. Defaults to "cuda".

    Returns:
        numpy array [len(dataset) x num_classes]: Predictions.
    """
    # Use best model
    model.load_state_dict(torch.load(latest_model))
    
    model.eval()

    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS
    )
    
    preds = []
    with torch.no_grad():
        for data in loader:
            pred = model(data[0].to(device)).squeeze(-1)
            preds.append(pred.detach().cpu().numpy())

    preds = np.concatenate(preds, 0)
    return preds

# GroupKFold

In [ ]:
from sklearn.model_selection import GroupKFold

def k_fold(config, df, df_test):
    """
    Performs a patient grouped k-fold cross validation.
    """

    pred_oof = np.zeros(len(df))
    preds_test = []
    
    gkf = GroupKFold(n_splits=config.k)
    splits = list(gkf.split(X=df, y=df, groups=df["breath_id"]))

    for i, (train_idx, val_idx) in enumerate(splits):
        if i in config.selected_folds:
            print(f"\n-------------   Fold {i + 1} / {config.k}  -------------\n")

            df_train = df.iloc[train_idx].copy().reset_index(drop=True)
            df_val = df.iloc[val_idx].copy().reset_index(drop=True)

            pred_val, pred_test = train(config, df_train, df_val, df_test, i)
            
            pred_oof[val_idx] = pred_val.flatten()
            preds_test.append(pred_test.flatten())

    print(f'\n -> CV MAE : {compute_metric(df, pred_oof) :.3f}')

    return pred_oof, np.mean(preds_test, 0)

# Training and predictions

In [ ]:
print("Ready")

In [ ]:
pred_oof, pred_test = k_fold(
    Config, 
    df_train,
    df_test,
)

# Debug

In [ ]:
# pred_oof = np.zeros(len(df_train))
# preds_test = []

# gkf = GroupKFold(n_splits=Config.k)
# splits = list(gkf.split(X=df_train, y=df_train, groups=df_train["breath_id"]))

In [ ]:
# df_train_deb = df_train.iloc[splits[0][0]].copy().reset_index(drop=True)
# df_val_deb = df_train.iloc[splits[0][1]].copy().reset_index(drop=True)

# print(df_train_deb.shape)
# print(df_val_deb.shape)

In [ ]:
# pred_val, pred_test = train(config, df_train, df_val, df_test, i)

In [ ]:
# seed_everything(Config.seed)

# model = RNNModel(
#     input_dim=Config.input_dim,
#     lstm_dim=Config.lstm_dim,
#     dense_dim=Config.dense_dim,
#     logit_dim=Config.logit_dim,
#     num_classes=Config.num_classes,
# ).to(Config.device)
# model.zero_grad()

In [ ]:
# if "pressure" not in df_train_deb.columns:
#     df_train_deb['pressure'] = 0

# df_train_deb_grp = df_train_deb.groupby('breath_id').agg(list).reset_index()

In [ ]:
# train_dataset = VentilatorDataset(df_train_deb)

In [ ]:


# train_dataset = VentilatorDataset(df_train_deb)
# val_dataset = VentilatorDataset(df_val_deb)
# test_dataset = VentilatorDataset(df_test)

# n_parameters = count_parameters(model)

# print(f"    -> {len(train_dataset)} training breathes")
# print(f"    -> {len(val_dataset)} validation breathes")
# print(f"    -> {n_parameters} trainable parameters\n")

# Notes:

- Best ways to minimize time for model eval
- Try 10 folds

- Plot train/test losses
- Early stopping
- wandb

- Feature engineering

- Prediction postprocessing (round to discrete steps)
- Median based fold prediction


- Build public notebook in Pytorch (CDeotte)

- TPU build



# Create submission

In [ ]:
sub['pressure'] = pred_test
sub.to_csv('submission.csv', index=False)